# Step 2c: explaining the text models

**Models (3):** small transformer · word2vec + logistic regression · doc2vec + logistic regression.

**Unit of explanation: one post** (a short sequence of about 10 words). For each post we ask which **words** pushed the model's score for "inauthentic" up or down.

**Two explanation methods per model:** SHAP (KernelSHAP over the post's unique words) and LIME (text, bag-of-words mode). Both treat the model as a black box: they delete words from the post and watch the score. A deleted word is removed entirely (all its occurrences), in both methods, so the two are asking the same question.

**The answer key and the decoy**

* *Marker words* (`giveaway`, `promo`, `idiot`, `sheep`, ...) are planted in 35% of inauthentic accounts' posts. A good explanation of a post that contains one should rank it first.
* *The decoy token `#trending`* appears in 70% of inauthentic posts in train and validation, but only 10% in test. A model that leans on it is using a shortcut that breaks on new data. A good explainer should reveal that, so we track the credit it gives `#trending` separately and compare it with what the model really does when the token is deleted.

**Three groups of posts are explained** (the same posts for every model, all from the test split):

| Stratum | Posts | Why |
|---|---|---|
| `marker` | contain a planted marker word | do the explanations find the real signal? |
| `spurious_only` | contain `#trending` and no marker | does the model's reliance on the decoy show up? |
| `neutral` | contain neither | what do explanations say when there is nothing to find? |

**Criteria:** correctness (marker is the top word; share of credit on marker words), faithfulness (delete the top words vs random words), agreement (SHAP vs LIME), stability (drop one filler word and re-explain; a seed-repeat control). See `docs/step2_plan.md`.

**A caveat on word2vec and doc2vec.** Step 1 trained their classifiers on whole accounts (6 posts averaged). Here we feed them a single post, which is a different input than they were trained on, so their scores for a single post are noisier than for an account. The explanation is still a faithful description of what the model does with that input.

> Run top to bottom, one cell at a time. `QUICK = True` gives a 2-minute smoke test. A full run takes 10 to 20 minutes (doc2vec dominates).

## 0. Setup

In [ ]:
import sys
import time
import warnings
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))  # so `import xai_eval` works from notebooks/

import xai_eval  # noqa: F401  (imports xgboost / lightgbm before torch: macOS OpenMP workaround)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from xai_eval import data as xd
from xai_eval import io as xio
from xai_eval import loaders as L
from xai_eval import metrics as xm

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 160)
torch.set_num_threads(4)

import shap
from lime.lime_text import LimeTextExplainer
from sklearn.metrics import roc_auc_score

In [ ]:
SEED = 0
QUICK = False

N_PER_STRATUM = 5 if QUICK else 60       # posts explained per stratum (x3)
N_STAB_PER_STRATUM = 3 if QUICK else 20  # subset for the stability checks
SHAP_NSAMPLES = 100 if QUICK else 1000   # KernelSHAP coalitions per post (exact if the post has <= 10 unique words)
LIME_NSAMPLES = 150 if QUICK else 1000   # LIME perturbed texts per post
KS = [1, 2, 3]                           # words deleted in the faithfulness test
N_RANDOM = 5
FAMILY = "text"
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1. Data, the sample of posts, and the answer key

In [ ]:
D = xd.load_data(REPO_ROOT)
MARKERS, SPUR = D.marker_set, D.spurious_token

tok_sets = [set(t) for t in D.post_tokens]
has_marker = np.array([bool(MARKERS & s) for s in tok_sets])    # ANSWER KEY (from truth/ground_truth.json)
has_spur = np.array([SPUR in s for s in tok_sets])
in_test = D.is_te[D.post_account]
post_label = D.y[D.post_account]

STRATA = {
    "marker": in_test & has_marker,
    "spurious_only": in_test & has_spur & ~has_marker,
    "neutral": in_test & ~has_marker & ~has_spur,
}
rng = np.random.default_rng(SEED)
POST_IDX, STRATUM = [], []
for s, m in STRATA.items():
    pick = rng.choice(np.flatnonzero(m), size=N_PER_STRATUM, replace=False)
    POST_IDX += list(pick)
    STRATUM += [s] * len(pick)
POST_IDX, STRATUM = np.array(POST_IDX), np.array(STRATUM)
TOKS = [D.post_tokens[i] for i in POST_IDX]
STAB_POS = np.concatenate([np.flatnonzero(STRATUM == s)[:N_STAB_PER_STRATUM] for s in STRATA])

print({s: int(m.sum()) for s, m in STRATA.items()}, "test posts available per stratum")
print("share inauthentic in each sampled stratum:", {s: round(float(post_label[POST_IDX[STRATUM == s]].mean()), 2) for s in STRATA})
print("example (marker):", " ".join(TOKS[0]))
print("example (spurious only):", " ".join(TOKS[N_PER_STRATUM]))

## 2. Reload the step-1 models and check them

In [ ]:
MODELS, SCORERS, CHECKS = {}, {}, []
for name in L.TEXT_MODELS:
    MODELS[name] = L.load_model(name, D)
    SCORERS[name] = L.TextScorer(name, MODELS[name], D)     # list of token lists -> P(inauthentic) per post
    # doc2vec cannot be reproduced exactly (its inference is seeded by a per-process hash), so it gets a rank check
    CHECKS.append(L.verify_reload(name, D, SCORERS[name].account_scores(D), loose=(name == "doc2vec_logreg")))
assert all(c["status"] == "OK" for c in CHECKS), "A reloaded model does not match its step-1 predictions"

TEST_POSTS = np.flatnonzero(in_test)
SCORE_SD = {n: float(np.std(SCORERS[n]([D.post_tokens[i] for i in TEST_POSTS]))) for n in MODELS}  # scale for faithfulness

`doc2vec` note: gensim seeds doc2vec's inference with Python's `hash()`, which changes with every Python process, so the same post would get a slightly different vector on every run. `xai_eval.loaders` swaps that for a fixed hash so the model is a repeatable function (explanations of a function that changes between calls would be meaningless). The weights are untouched; step-1 scores can therefore only be matched by rank (correlation above 0.95), not exactly.

## 3. Reliance probe: what does each model actually use?

On all test posts (the post inherits its account's label), we compute the AUC three ways: as is, with `#trending` stripped, and with the marker words stripped. In the test split the decoy is uninformative (it appears in about 10% of posts of both classes), so a model that learned to lean on it in training should *gain* AUC when it is removed (a negative `drop_without_decoy`), while a model that leans on the real signal loses AUC when the markers are removed (a positive `drop_without_markers`). This is what the explanations should reveal.

In [ ]:
def strip(tokens, words):
    return [t for t in tokens if t not in words]


te_tok = [D.post_tokens[i] for i in TEST_POSTS]
yt = post_label[TEST_POSTS]
rows = []
for name, sc in SCORERS.items():
    rows.append(dict(
        model=name,
        auc_original=roc_auc_score(yt, sc(te_tok)),
        auc_without_decoy=roc_auc_score(yt, sc([strip(t, {SPUR}) for t in te_tok])),
        auc_without_markers=roc_auc_score(yt, sc([strip(t, MARKERS) for t in te_tok])),
    ))
RELIANCE = pd.DataFrame(rows).set_index("model")
RELIANCE["drop_without_decoy"] = RELIANCE.auc_original - RELIANCE.auc_without_decoy
RELIANCE["drop_without_markers"] = RELIANCE.auc_original - RELIANCE.auc_without_markers
display(RELIANCE.round(3))

## 4. Explanation and evaluation code

For a post with words `w_1 .. w_M` (unique words, in order of first appearance):

* **SHAP:** `shap.KernelExplainer` on a 0/1 vector that says which words are kept. "All words kept" is the explained input; "no words kept" is the reference. The attribution of word *j* is its Shapley value, and the values add up to (score of the full post) minus (score of the empty post).
* **LIME:** `LimeTextExplainer` with bag-of-words perturbation. It fits a weighted linear model on the score of many randomly thinned versions of the post; the weight of word *j* is its attribution.

Positive = pushes the score towards "inauthentic".

In [ ]:
def players(tokens):
    return list(dict.fromkeys(tokens))           # unique words, order of first appearance


def keep_words(tokens, words, z):
    kept = {w for w, flag in zip(words, z) if flag}
    return [t for t in tokens if t in kept]


def explain_shap(scorer, tokens, seed=SEED):
    words = players(tokens)
    M = len(words)
    f = lambda Z: scorer([keep_words(tokens, words, z) for z in np.atleast_2d(Z)])  # noqa: E731
    np.random.seed(seed)
    ex = shap.KernelExplainer(f, np.zeros((1, M)))
    return words, np.asarray(ex.shap_values(np.ones((1, M)), nsamples=SHAP_NSAMPLES, silent=True)).reshape(-1)


def explain_lime(scorer, tokens, seed=SEED):
    words = players(tokens)

    def proba(texts):
        p = scorer([t.split() for t in texts])
        return np.c_[1 - p, p]

    ex = LimeTextExplainer(split_expression=r"\s+", bow=True, random_state=seed)
    e = ex.explain_instance(" ".join(tokens), proba, labels=(1,), num_features=len(words), num_samples=LIME_NSAMPLES)
    w = dict(e.as_list(label=1))
    return words, np.array([w.get(x, 0.0) for x in words])


EXPLAINERS = {"SHAP": explain_shap, "LIME": explain_lime}


def explain_all(name, token_lists, seed=SEED):
    """{method: list of attribution vectors} and seconds per post for each method."""
    out, secs = {}, {}
    for m, fn in EXPLAINERS.items():
        t0 = time.time()
        out[m] = [fn(SCORERS[name], toks, seed)[1] for toks in token_lists]
        secs[m] = (time.time() - t0) / len(token_lists)
    return out, secs


def pad(vectors):
    d = max(len(v) for v in vectors)
    return np.stack([np.pad(v, (0, d - len(v))) for v in vectors])


def remove_words(tokens, idx):
    drop = {players(tokens)[j] for j in idx}
    return [t for t in tokens if t not in drop]

In [ ]:
SC = xm.Scorecard("post_text")
INST_ROWS, ATTR_STORE = [], {}


def evaluate(name, attrs, secs):
    scorer = SCORERS[name]
    words = [players(t) for t in TOKS]
    n_items = np.array([len(w) for w in words])
    mk = np.flatnonzero(STRATUM == "marker")
    sp = np.flatnonzero(STRATUM == "spurious_only")
    for m, vecs in attrs.items():
        # ---- correctness on posts with a planted marker
        top1, mass = [], []
        for i in mk:
            a, w = vecs[i], words[i]
            is_m = np.array([x in MARKERS for x in w])
            top1.append(float(is_m[int(np.argmax(a))]))               # signed: the marker should be the strongest push UP
            mass.append(np.abs(a)[is_m].sum() / max(np.abs(a).sum(), 1e-12))
        chance = np.mean([np.mean([x in MARKERS for x in words[i]]) for i in mk])
        SC.add(name, m, "correctness_marker_top1", top1, f"chance about {1 / np.mean(n_items[mk]):.2f}")
        SC.add(name, m, "correctness_marker_mass", mass, f"chance = {chance:.2f} (share of the post's words that are markers)")
        # ---- shortcut diagnostics on posts that have only the decoy
        s_top1, s_mass = [], []
        for i in sp:
            a, w = vecs[i], words[i]
            j = w.index(SPUR)
            s_top1.append(float(int(np.argmax(np.abs(a))) == j))
            s_mass.append(abs(a[j]) / max(np.abs(a).sum(), 1e-12))
        SC.add(name, m, "shortcut_top1", s_top1, "diagnostic: share of decoy-only posts where #trending is the top word")
        SC.add(name, m, "shortcut_mass", s_mass, "diagnostic: share of |credit| given to #trending in decoy-only posts")
        # ---- faithfulness: delete the top words
        dele = xm.deletion_gain(scorer, TOKS, pad(vecs), remove_words, KS, SCORE_SD[name], n_random=N_RANDOM, seed=SEED, n_items=n_items)
        dele.assign(model=name, method=m).to_csv(xio.out_dir(D, FAMILY) / f"deletion_{name}_{m}.csv", index=False)
        faith = xm.mean_gain_per_instance(dele)
        SC.add(name, m, "faithfulness", faith)
        SC.add(name, m, "seconds_per_explanation", secs[m])
        for i in range(len(TOKS)):
            INST_ROWS.append(dict(model=name, method=m, post_id=int(POST_IDX[i]), stratum=STRATUM[i], faithfulness=faith[i]))
    # ---- what the MODEL does when the decoy is deleted (compare with the credit above)
    base = scorer([TOKS[i] for i in sp])
    after = scorer([strip(TOKS[i], {SPUR}) for i in sp])
    SC.add(name, "model (no explainer)", "shortcut_model_effect", np.abs(base - after) / SCORE_SD[name],
           "diagnostic: |score change| when #trending is deleted, in score std units")
    # ---- agreement
    a, b = attrs["SHAP"], attrs["LIME"]
    SC.add(name, "SHAP vs LIME", "agreement_spearman", [xm.rank_corr(np.abs(x), np.abs(y)) for x, y in zip(a, b)])
    SC.add(name, "SHAP vs LIME", "agreement_top3", [xm.topk_overlap(x, y, 3) for x, y in zip(a, b)])


def stability_checks(name, attrs):
    """Drop one random filler word (not a marker, not the decoy), re-explain, compare on the words that remain."""
    r = np.random.default_rng(SEED + 7)
    pert, keep_cols = [], []
    for i in STAB_POS:
        w = players(TOKS[i])
        filler = [j for j, x in enumerate(w) if x not in MARKERS and x != SPUR]
        j = int(r.choice(filler))
        pert.append([t for t in TOKS[i] if t != w[j]])
        keep_cols.append([c for c in range(len(w)) if c != j])
    for m, fn in EXPLAINERS.items():
        noisy = [fn(SCORERS[name], p, SEED)[1] for p in pert]
        again = [fn(SCORERS[name], TOKS[i], SEED + 100)[1] for i in STAB_POS]
        s_noise = [xm.rank_corr(np.abs(attrs[m][i][cols]), np.abs(v)) for i, cols, v in zip(STAB_POS, keep_cols, noisy)]
        s_again = [xm.rank_corr(np.abs(attrs[m][i]), np.abs(v)) for i, v in zip(STAB_POS, again)]
        SC.add(name, m, "stability_drop_word", s_noise)
        SC.add(name, m, "repeatability_seed", s_again)


def show_posts(name, n_each=1):
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
    for ax, s in zip(axes, ("marker", "spurious_only")):
        i = int(np.flatnonzero(STRATUM == s)[0])
        w = players(TOKS[i])
        x = np.arange(len(w))
        for k, (m, col) in enumerate([("SHAP", "#4C78A8"), ("LIME", "#F58518")]):
            ax.bar(x + (k - 0.5) * 0.4, ATTR_STORE[name][m][i], 0.4, color=col, label=m)
        ax.set_xticks(x)
        labels = ax.set_xticklabels(w, rotation=60, ha="right")
        for lab, word in zip(labels, w):
            if word in MARKERS:
                lab.set_color("#E45756")
            elif word == SPUR:
                lab.set_color("#B8860B")
        ax.axhline(0, color="gray", lw=0.8)
        ax.set_title(f"{name} | {s} post (red = marker, gold = #trending)", fontsize=9)
    axes[0].legend(frameon=False)
    plt.tight_layout()
    plt.show()


def run_model(name):
    print(f"=== {name} ===")
    attrs, secs = explain_all(name, TOKS)
    print("  seconds per post:", {k: round(v, 2) for k, v in secs.items()})
    ATTR_STORE[name] = attrs
    evaluate(name, attrs, secs)
    stability_checks(name, attrs)
    recs = []
    for m, vecs in attrs.items():
        for i, v in enumerate(vecs):
            for word, a in zip(players(TOKS[i]), v):
                recs.append(dict(model=name, method=m, post_id=int(POST_IDX[i]), stratum=STRATUM[i], word=word, attribution=float(a)))
    xio.save_table(D, FAMILY, f"attributions_{name}", pd.DataFrame(recs))
    keep = ["correctness_marker_top1", "correctness_marker_mass", "shortcut_mass", "faithfulness", "agreement_spearman",
            "stability_drop_word", "repeatability_seed"]
    display(SC.df[(SC.df.model == name) & SC.df.criterion.isin(keep)].pivot_table(index="method", columns="criterion", values="value").round(3))
    show_posts(name)

## 5. Transformer

A 2-layer transformer encoder trained from scratch on the posts, with a [CLS] token whose output is classified. Word order matters to such a model in principle, but in this data the planted signal is a single word, so a good explanation is simply "the marker word". The decoy `#trending` was reliable in training, so the reliance probe tells us how much the model learned to use it.

In [ ]:
run_model('transformer')

## 6. word2vec + logistic regression

Each word has a fixed 50-dimensional vector; a post is the **average** of its word vectors, and a logistic regression sits on top. Because of the averaging, the effect of one word on the score depends on how many other words there are (a marker in a 6-word post counts for more than in a 14-word post). Word-level attributions are still well defined by deleting the word and re-averaging.

In [ ]:
run_model('word2vec_logreg')

## 7. doc2vec + logistic regression

doc2vec learns a vector per *document* and, for a new post, **infers** one by running a few training passes with the weights frozen. That inference is an optimisation, not a formula, so deleting one word can change the vector in non-obvious ways. Expect this model to be the hardest to explain faithfully, and slow: each score needs 30 inference passes (this section dominates the notebook's run time).

In [ ]:
run_model('doc2vec_logreg')

## 8. Scorecard for the text group

In [ ]:
tbl = SC.table()
order = ["correctness_marker_top1", "correctness_marker_mass", "faithfulness", "agreement_spearman", "agreement_top3",
         "stability_drop_word", "repeatability_seed", "shortcut_top1", "shortcut_mass", "shortcut_model_effect",
         "seconds_per_explanation"]
tbl = tbl[[c for c in order if c in tbl.columns]]
display(tbl.round(3))
print("Reliance probe (post-level AUC):")
display(RELIANCE.round(3))

In [ ]:
df = SC.df
crits = ["correctness_marker_top1", "correctness_marker_mass", "faithfulness", "stability_drop_word"]
fig, axes = plt.subplots(1, 4, figsize=(17, 3.6), sharey=True)
rows = [(m, meth) for m in L.TEXT_MODELS for meth in ("SHAP", "LIME")]
for ax, c in zip(axes, crits):
    d = df[df.criterion == c].set_index(["model", "method"]).reindex(rows)
    y = np.arange(len(rows))
    ax.barh(y, d["value"], 0.6, color=["#4C78A8" if r[1] == "SHAP" else "#F58518" for r in rows])
    ax.errorbar(d["value"], y, xerr=[(d["value"] - d["ci_low"]).clip(lower=0), (d["ci_high"] - d["value"]).clip(lower=0)],
                fmt="none", ecolor="black", lw=0.8, capsize=2)
    ax.set_title(c.replace("_", " "))
    ax.axvline(0, color="gray", lw=0.8)
axes[0].set_yticks(np.arange(len(rows)))
axes[0].set_yticklabels([f"{m} / {meth}" for m, meth in rows], fontsize=8)
axes[0].invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Does the credit given to the decoy match what the model actually does with it?
d = df[df.criterion.isin(["shortcut_mass", "shortcut_model_effect"])].pivot_table(index=["model", "method"], columns="criterion", values="value")
display(d.round(3))

Reading the last table: `shortcut_model_effect` is a fact about the model (how much its score moves when `#trending` is deleted, in units of the score's standard deviation). `shortcut_mass` is what the explainer says (share of credit on `#trending`). An explainer that is faithful about the decoy gives it a lot of credit exactly when the model effect is large.

In [ ]:
xio.save_scorecard(D, FAMILY, SC, reload_checks=CHECKS, meta=dict(
    seed=SEED, quick=QUICK, n_per_stratum=N_PER_STRATUM, shap_nsamples=SHAP_NSAMPLES, lime_nsamples=LIME_NSAMPLES, ks=KS))
xio.save_table(D, FAMILY, "reliance_probe", RELIANCE.reset_index())
pd.DataFrame(INST_ROWS).to_csv(xio.out_dir(D, FAMILY) / "per_instance_scores.csv", index=False)
tbl.round(4).to_csv(xio.out_dir(D, FAMILY) / "scorecard_wide.csv")

## 9. What to take from this notebook

1. Do the explanations put the marker word first (`correctness_marker_top1`), and is that better than chance?
2. Which models lean on the decoy (reliance probe), and do SHAP and LIME both show it (`shortcut_mass` against `shortcut_model_effect`)?
3. Which model's explanations are least faithful, and does that match the intuition that doc2vec's inference is not a simple function of its words?
4. Do SHAP and LIME agree? When they disagree, which of the two is more faithful?
5. How much of the instability is sampling noise (`repeatability_seed`) rather than a sensitivity of the model (`stability_drop_word`)?